In [11]:
# Block 1: Setup and Google Drive Mount

# Mount Google Drive FIRST
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive
DRIVE_DIR = "/content/drive/MyDrive/SmartVisionAI"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/model_output", exist_ok=True)

!pip install roboflow -q
!pip install tensorflow==2.15.0 -q

import json
import shutil
import random
import hashlib
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pickle

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU Found:", gpus[0].name)
else:
    print("No GPU found")
print(f"Drive folder: {DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.15.0
TensorFlow Version: 2.19.0
GPU Found: /physical_device:GPU:0
Drive folder: /content/drive/MyDrive/SmartVisionAI


In [12]:
# Block 2: Configuration

IMG_SIZE = 640
NUM_CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 60
LEARNING_RATE = 0.001
NUM_CLASSES = 10
MAX_BOXES = 50

CLASS_NAMES = [
    "sunglasses", "knife", "water_bottle", "pen", "chair",
    "human_face", "mobile_phone", "helmet", "fire", "can"
]

CLASS_WEIGHTS = [1.41, 1.22, 1.27, 0.75, 1.02, 1.11, 1.44, 1.10, 0.74, 0.66]

# Paths (using Drive)
MERGED_DIR = f"{DRIVE_DIR}/merged_dataset"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"
OUTPUT_DIR = f"{DRIVE_DIR}/model_output"

print("Configuration loaded")
print(f"Data will be saved to: {DRIVE_DIR}")

Configuration loaded
Data will be saved to: /content/drive/MyDrive/SmartVisionAI


In [13]:
# Block 3: Download Datasets

from roboflow import Roboflow

API_KEY = "0Wgp8GjEQCPnJbk4eVeF"
rf = Roboflow(api_key=API_KEY)

datasets_info = [
    {"name": "sunglasses", "project": "sunglasses-tvknx-wlwcu", "version": 1,
     "target_class": "sunglasses", "source_classes": ["Sunglasses", "sunglasses"], "ignore_classes": []},
    {"name": "knife", "project": "knifedetectionv4-6pfwh", "version": 3,
     "target_class": "knife", "source_classes": ["knife-KyFz", "knife", "knives"], "ignore_classes": []},
    {"name": "water_bottle", "project": "water-bottle-qginj-6glhq", "version": 4,
     "target_class": "water_bottle", "source_classes": ["Water-bottle", "waterbottle"], "ignore_classes": []},
    {"name": "pen", "project": "pen-kkqzs-tvftk", "version": 3,
     "target_class": "pen", "source_classes": ["p", "Pen"], "ignore_classes": []},
    {"name": "chair", "project": "chair-walws-ziqgx", "version": 1,
     "target_class": "chair", "source_classes": ["chair"], "ignore_classes": ["table"]},
    {"name": "human_face", "project": "face-detection-mik1i-iuvar", "version": 1,
     "target_class": "human_face", "source_classes": ["faces", "face"], "ignore_classes": []},
    {"name": "mobile_phone", "project": "e-waste-mobile-phones-drlac", "version": 3,
     "target_class": "mobile_phone", "source_classes": ["Phones", "Mobile phone"], "ignore_classes": []},
    {"name": "helmet", "project": "helmet-motorcycle-detection-pro0q-ozzzn", "version": 3,
     "target_class": "helmet", "source_classes": ["helmet", "Helmet"], "ignore_classes": []},
    {"name": "fire", "project": "fire-8klur-fceqf", "version": 3,
     "target_class": "fire", "source_classes": ["fire"], "ignore_classes": ["smoke"]},
    {"name": "can", "project": "cans-avukd-b09pf", "version": 5,
     "target_class": "can", "source_classes": ["cans", "can"], "ignore_classes": ["metal"]}
]

# Download to local (faster), then merge to Drive
dataset_paths = {}
print("Downloading datasets...")
for info in datasets_info:
    print(f"  {info['name']}...")
    project = rf.workspace("yuyutsu-cxcso").project(info["project"])
    version = project.version(info["version"])
    dataset = version.download("coco")
    dataset_paths[info["name"]] = dataset.location

print("All datasets downloaded!")

  sunglasses...
loading Roboflow workspace...
loading Roboflow project...
  knife...
loading Roboflow workspace...
loading Roboflow project...
  water_bottle...
loading Roboflow workspace...
loading Roboflow project...
  pen...
loading Roboflow workspace...
loading Roboflow project...
  chair...
loading Roboflow workspace...
loading Roboflow project...
  human_face...
loading Roboflow workspace...
loading Roboflow project...
  mobile_phone...
loading Roboflow workspace...
loading Roboflow project...
  helmet...
loading Roboflow workspace...
loading Roboflow project...
  fire...
loading Roboflow workspace...
loading Roboflow project...
  can...
loading Roboflow workspace...
loading Roboflow project...
All datasets downloaded!


In [14]:
# Block 4: Merge Datasets (saves to Drive)

# Clear old merged data
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)

for split in ["train", "valid", "test"]:
    os.makedirs(f"{MERGED_DIR}/{split}/images", exist_ok=True)

def get_class_id(class_name, dataset_info):
    class_name_lower = class_name.lower()
    for ignore in dataset_info["ignore_classes"]:
        if ignore.lower() == class_name_lower:
            return -1
    for source in dataset_info["source_classes"]:
        if source.lower() == class_name_lower:
            return CLASS_NAMES.index(dataset_info["target_class"])
    return -1

def get_short_filename(dataset_name, original_filename, img_id):
    ext = os.path.splitext(original_filename)[1]
    if not ext:
        ext = ".jpg"
    short_hash = hashlib.md5(original_filename.encode()).hexdigest()[:8]
    return f"{dataset_name}_{img_id}_{short_hash}{ext}"

def merge_dataset(dataset_info, split):
    dataset_path = dataset_paths[dataset_info["name"]]
    ann_file = f"{dataset_path}/{split}/_annotations.coco.json"

    if not os.path.exists(ann_file):
        return []

    with open(ann_file, 'r') as f:
        coco_data = json.load(f)

    orig_id_to_name = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
    image_id_to_filename = {}
    processed_anns = []

    for img in coco_data["images"]:
        src = f"{dataset_path}/{split}/{img['file_name']}"
        new_name = get_short_filename(dataset_info["name"], img['file_name'], img['id'])
        dst = f"{MERGED_DIR}/{split}/images/{new_name}"

        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
        if os.path.exists(dst):
            image_id_to_filename[img["id"]] = new_name

    for ann in coco_data["annotations"]:
        orig_name = orig_id_to_name.get(ann["category_id"], "")
        new_id = get_class_id(orig_name, dataset_info)
        if new_id == -1:
            continue
        new_filename = image_id_to_filename.get(ann["image_id"])
        if new_filename:
            processed_anns.append({"filename": new_filename, "class_id": new_id, "bbox": ann["bbox"]})

    return processed_anns

print("Merging datasets to Drive...")
for split in ["train", "valid", "test"]:
    all_anns = []
    for info in datasets_info:
        anns = merge_dataset(info, split)
        all_anns.extend(anns)

    image_dir = f"{MERGED_DIR}/{split}/images"
    image_files = os.listdir(image_dir) if os.path.exists(image_dir) else []

    images_list = []
    filename_to_id = {}
    for idx, fname in enumerate(image_files):
        img = cv2.imread(f"{image_dir}/{fname}")
        if img is not None:
            h, w = img.shape[:2]
            images_list.append({"id": idx, "file_name": fname, "width": w, "height": h})
            filename_to_id[fname] = idx

    annotations_list = []
    for i, ann in enumerate(all_anns):
        if ann["filename"] in filename_to_id:
            annotations_list.append({
                "id": i, "image_id": filename_to_id[ann["filename"]],
                "category_id": ann["class_id"], "bbox": ann["bbox"],
                "area": ann["bbox"][2] * ann["bbox"][3], "iscrowd": 0
            })

    coco_output = {
        "images": images_list, "annotations": annotations_list,
        "categories": [{"id": i, "name": n} for i, n in enumerate(CLASS_NAMES)]
    }
    with open(f"{MERGED_DIR}/{split}/annotations.json", 'w') as f:
        json.dump(coco_output, f)

    print(f"  {split}: {len(images_list)} images, {len(annotations_list)} annotations")

print(f"Merge complete! Saved to: {MERGED_DIR}")

Merging datasets to Drive...
  train: 27841 images, 39463 annotations
  valid: 2614 images, 3229 annotations
  test: 1782 images, 2358 annotations
Merge complete! Saved to: /content/drive/MyDrive/SmartVisionAI/merged_dataset


In [15]:
# Block 5: Verify Dataset

print("Verifying merged dataset...")

for split in ["train", "valid", "test"]:
    ann_file = f"{MERGED_DIR}/{split}/annotations.json"
    with open(ann_file, 'r') as f:
        data = json.load(f)

    num_images = len(data["images"])
    num_anns = len(data["annotations"])

    class_counts = {i: 0 for i in range(NUM_CLASSES)}
    for ann in data["annotations"]:
        class_counts[ann["category_id"]] += 1

    print(f"\n{split.upper()}: {num_images} images, {num_anns} annotations")
    for i, count in class_counts.items():
        print(f"  {CLASS_NAMES[i]}: {count}")

print("\nVerification complete!")

Verifying merged dataset...

TRAIN: 27841 images, 39463 annotations
  sunglasses: 2190
  knife: 2425
  water_bottle: 3579
  pen: 4948
  chair: 1878
  human_face: 4630
  mobile_phone: 1918
  helmet: 6703
  fire: 7230
  can: 3962

VALID: 2614 images, 3229 annotations
  sunglasses: 268
  knife: 187
  water_bottle: 329
  pen: 706
  chair: 211
  human_face: 49
  mobile_phone: 458
  helmet: 130
  fire: 673
  can: 218

TEST: 1782 images, 2358 annotations
  sunglasses: 122
  knife: 111
  water_bottle: 57
  pen: 332
  chair: 206
  human_face: 263
  mobile_phone: 207
  helmet: 115
  fire: 789
  can: 156

Verification complete!


In [16]:
# Block 5b: Copy Dataset to Local Storage for FAST Training

LOCAL_MERGED = "/content/merged_local"

if not os.path.exists(LOCAL_MERGED):
    print("Copying dataset to local Colab storage...")
    print("⏱ This takes 2-3 minutes but saves HOURS!")
    shutil.copytree(MERGED_DIR, LOCAL_MERGED)
    print("Dataset copied to LOCAL storage!")
else:
    print("Local dataset already exists")

# Update the merged directory path
MERGED_DIR = LOCAL_MERGED
print(f"New data location: {MERGED_DIR}")

Copying dataset to local Colab storage...
⏱ This takes 2-3 minutes but saves HOURS!
Dataset copied to LOCAL storage!
New data location: /content/merged_local


In [17]:
# Block 6: Data Loading Functions

def load_dataset(split):
    ann_file = f"{MERGED_DIR}/{split}/annotations.json"
    with open(ann_file, 'r') as f:
        coco_data = json.load(f)

    image_info = {img["id"]: img for img in coco_data["images"]}
    anns_by_image = {}
    for ann in coco_data["annotations"]:
        if ann["image_id"] not in anns_by_image:
            anns_by_image[ann["image_id"]] = []
        anns_by_image[ann["image_id"]].append(ann)

    dataset = []
    for img_id, img_info in image_info.items():
        img_path = f"{MERGED_DIR}/{split}/images/{img_info['file_name']}"
        if not os.path.exists(img_path):
            continue

        anns = anns_by_image.get(img_id, [])
        boxes = []
        class_ids = []

        for ann in anns:
            bbox = ann["bbox"]
            x_center = (bbox[0] + bbox[2] / 2) / img_info["width"]
            y_center = (bbox[1] + bbox[3] / 2) / img_info["height"]
            w = bbox[2] / img_info["width"]
            h = bbox[3] / img_info["height"]

            x_center = max(0, min(1, x_center))
            y_center = max(0, min(1, y_center))
            w = max(0, min(1, w))
            h = max(0, min(1, h))

            if w > 0.001 and h > 0.001:
                boxes.append([x_center, y_center, w, h])
                class_ids.append(ann["category_id"])

        dataset.append({"image_path": img_path, "boxes": boxes, "class_ids": class_ids})

    return dataset

def load_and_preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype(np.float32) / 255.0

def prepare_targets(boxes, class_ids):
    num_boxes = min(len(boxes), MAX_BOXES)
    boxes_arr = np.zeros((MAX_BOXES, 4), dtype=np.float32)
    class_arr = np.zeros((MAX_BOXES,), dtype=np.int32)
    for i in range(num_boxes):
        boxes_arr[i] = boxes[i]
        class_arr[i] = class_ids[i]
    return boxes_arr, class_arr, num_boxes

print("Loading datasets...")
train_data = load_dataset("train")
valid_data = load_dataset("valid")
test_data = load_dataset("test")
print(f"Train: {len(train_data)}, Valid: {len(valid_data)}, Test: {len(test_data)}")

Loading datasets...
Train: 27841, Valid: 2614, Test: 1782


In [18]:
# Block 7: Data Generator

class DataGenerator(keras.utils.Sequence):
    def __init__(self, dataset, batch_size, augment=False, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = list(range(len(dataset)))
        if shuffle:
            random.shuffle(self.indices)

    def __len__(self):
        return len(self.dataset) // self.batch_size

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        batch_images = np.zeros((self.batch_size, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        batch_boxes = np.zeros((self.batch_size, MAX_BOXES, 4), dtype=np.float32)
        batch_classes = np.zeros((self.batch_size, MAX_BOXES), dtype=np.int32)
        batch_num_boxes = np.zeros((self.batch_size,), dtype=np.int32)

        for i, data_idx in enumerate(batch_indices):
            sample = self.dataset[data_idx]
            img = load_and_preprocess_image(sample["image_path"])
            boxes = [b[:] for b in sample["boxes"]]

            if self.augment and random.random() > 0.5:
                img = np.fliplr(img).copy()
                boxes = [[1.0 - b[0], b[1], b[2], b[3]] for b in boxes]

            boxes_arr, class_arr, num = prepare_targets(boxes, sample["class_ids"])
            batch_images[i] = img
            batch_boxes[i] = boxes_arr
            batch_classes[i] = class_arr
            batch_num_boxes[i] = num

        return batch_images, {"boxes": batch_boxes, "classes": batch_classes, "num_boxes": batch_num_boxes}

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.indices)

train_gen = DataGenerator(train_data, BATCH_SIZE, augment=True, shuffle=True)
valid_gen = DataGenerator(valid_data, BATCH_SIZE, augment=False, shuffle=False)
print(f"Train batches: {len(train_gen)}, Valid batches: {len(valid_gen)}")

Train batches: 1740, Valid batches: 163


In [19]:
# Block 8: Model Architecture

def conv_block(x, filters, kernel=3, strides=1, name=None):
    x = layers.Conv2D(filters, kernel, strides=strides, padding='same', use_bias=False, name=f"{name}_conv" if name else None)(x)
    x = layers.BatchNormalization(name=f"{name}_bn" if name else None)(x)
    x = layers.ReLU(name=f"{name}_relu" if name else None)(x)
    return x

def residual_block(x, filters, name=None):
    shortcut = x
    x = conv_block(x, filters, kernel=1, name=f"{name}_1")
    x = conv_block(x, filters, kernel=3, name=f"{name}_2")
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', name=f"{name}_skip")(shortcut)
    x = layers.Add(name=f"{name}_add")([x, shortcut])
    x = layers.ReLU(name=f"{name}_out")(x)
    return x

def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input")

    x = conv_block(inputs, 32, strides=2, name="stem1")
    x = conv_block(x, 64, strides=2, name="stem2")

    x = residual_block(x, 64, name="stage1")
    x = conv_block(x, 128, strides=2, name="down1")

    x = residual_block(x, 128, name="stage2a")
    x = residual_block(x, 128, name="stage2b")
    small = x

    x = conv_block(x, 256, strides=2, name="down2")
    x = residual_block(x, 256, name="stage3a")
    x = residual_block(x, 256, name="stage3b")
    medium = x

    x = conv_block(x, 512, strides=2, name="down3")
    x = residual_block(x, 512, name="stage4a")
    x = residual_block(x, 512, name="stage4b")
    large = x

    p5 = conv_block(large, 256, kernel=1, name="p5")
    p5_up = layers.UpSampling2D(2)(p5)

    p4 = conv_block(medium, 256, kernel=1, name="p4_lat")
    p4 = layers.Add()([p4, p5_up])
    p4 = conv_block(p4, 256, name="p4")
    p4_up = layers.UpSampling2D(2)(p4)

    p3 = conv_block(small, 256, kernel=1, name="p3_lat")
    p3 = layers.Add()([p3, p4_up])
    p3 = conv_block(p3, 256, name="p3")

    num_anchors = 3
    out_channels = num_anchors * (5 + NUM_CLASSES)

    head_s = conv_block(p3, 256, name="head_s")
    head_s = layers.Dropout(0.2)(head_s)
    head_s = layers.Conv2D(out_channels, 1, name="detect_s")(head_s)

    head_m = conv_block(p4, 256, name="head_m")
    head_m = layers.Dropout(0.2)(head_m)
    head_m = layers.Conv2D(out_channels, 1, name="detect_m")(head_m)

    head_l = conv_block(p5, 256, name="head_l")
    head_l = layers.Dropout(0.2)(head_l)
    head_l = layers.Conv2D(out_channels, 1, name="detect_l")(head_l)

    return keras.Model(inputs, [head_s, head_m, head_l], name="SmartVision")

model = build_model()
model.summary()

# Verify output shapes
test_input = np.random.randn(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
outputs = model(test_input)
print(f"\nOutput shapes: small={outputs[0].shape}, medium={outputs[1].shape}, large={outputs[2].shape}")

Model: "SmartVision"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 640, 640,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem1_conv (Conv2D) │ (None, 320, 320,  │        864 │ input[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem1_bn            │ (None, 320, 320,  │        128 │ stem1_conv[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem1_relu (ReLU)   │ (None, 320, 320,  │          0 │ stem1_bn[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem2_conv (Conv2D) │ (None, 160, 160,  │     18,432 │ stem1_relu[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem2_bn            │ (None, 160, 160,  │        256 │ stem2_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem2_relu (ReLU)   │ (None, 160, 160,  │          0 │ stem2_bn[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_1_conv       │ (None, 160, 160,  │      4,096 │ stem2_relu[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_1_bn         │ (None, 160, 160,  │        256 │ stage1_1_conv[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_1_relu       │ (None, 160, 160,  │          0 │ stage1_1_bn[0][0] │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_2_conv       │ (None, 160, 160,  │     36,864 │ stage1_1_relu[0]… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_2_bn         │ (None, 160, 160,  │        256 │ stage1_2_conv[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_2_relu       │ (None, 160, 160,  │          0 │ stage1_2_bn[0][0] │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_add (Add)    │ (None, 160, 160,  │          0 │ stage1_2_relu[0]… │
│                     │ 64)               │            │ stem2_relu[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_out (ReLU)   │ (None, 160, 160,  │          0 │ stage1_add[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down1_conv (Conv2D) │ (None, 80, 80,    │     73,728 │ stage1_out[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down1_bn            │ (None, 80, 80,    │        512 │ down1_conv[0][0]

 Total params: 11,730,023 (44.75 MB)

 Trainable params: 11,716,519 (44.69 MB)

 Non-trainable params: 13,504 (52.75 KB)


Output shapes: small=(1, 80, 80, 45), medium=(1, 40, 40, 45), large=(1, 20, 20, 45)


In [20]:
# Block 9: FIXED Loss Function

ANCHORS = [
    [[0.02, 0.03], [0.04, 0.07], [0.08, 0.06]],
    [[0.07, 0.15], [0.15, 0.11], [0.14, 0.29]],
    [[0.28, 0.22], [0.38, 0.48], [0.90, 0.78]]
]
GRID_SIZES = [80, 40, 20]

def create_targets(batch_boxes, batch_classes, batch_num_boxes, scale_idx):
    batch_size = batch_boxes.shape[0]
    grid_size = GRID_SIZES[scale_idx]

    # Target arrays
    target_obj = np.zeros((batch_size, grid_size, grid_size, 3), dtype=np.float32)
    target_xy = np.zeros((batch_size, grid_size, grid_size, 3, 2), dtype=np.float32)
    target_wh = np.zeros((batch_size, grid_size, grid_size, 3, 2), dtype=np.float32)
    target_classes = np.zeros((batch_size, grid_size, grid_size, 3), dtype=np.int32)
    obj_mask = np.zeros((batch_size, grid_size, grid_size, 3), dtype=np.float32)

    for b in range(batch_size):
        for i in range(int(batch_num_boxes[b])):
            box = batch_boxes[b, i]
            cls = batch_classes[b, i]

            # Get box center and size
            cx, cy, bw, bh = box[0], box[1], box[2], box[3]

            # Skip invalid boxes
            if bw < 0.001 or bh < 0.001:
                continue

            # Find grid cell
            gx = int(cx * grid_size)
            gy = int(cy * grid_size)
            gx = min(max(gx, 0), grid_size - 1)
            gy = min(max(gy, 0), grid_size - 1)

            # Find best anchor
            best_anchor = 0
            best_iou = 0
            for a in range(3):
                anchor = ANCHORS[scale_idx][a]
                inter = min(bw, anchor[0]) * min(bh, anchor[1])
                union = bw * bh + anchor[0] * anchor[1] - inter
                iou = inter / (union + 1e-7)
                if iou > best_iou:
                    best_iou = iou
                    best_anchor = a

            # Calculate target xy (offset within cell, 0-1)
            tx = cx * grid_size - gx  # Offset within cell
            ty = cy * grid_size - gy

            # Calculate target wh (log scale relative to anchor)
            anchor_w, anchor_h = ANCHORS[scale_idx][best_anchor]
            tw = np.log(bw / anchor_w + 1e-7)
            th = np.log(bh / anchor_h + 1e-7)

            # Assign targets
            target_obj[b, gy, gx, best_anchor] = 1.0
            target_xy[b, gy, gx, best_anchor] = [tx, ty]
            target_wh[b, gy, gx, best_anchor] = [tw, th]
            target_classes[b, gy, gx, best_anchor] = cls
            obj_mask[b, gy, gx, best_anchor] = 1.0

    return target_obj, target_xy, target_wh, target_classes, obj_mask


def compute_loss(pred, target_obj, target_xy, target_wh, target_classes, obj_mask, scale_idx):
    batch_size = tf.shape(pred)[0]
    grid_size = GRID_SIZES[scale_idx]

    # Reshape prediction
    pred = tf.reshape(pred, [batch_size, grid_size, grid_size, 3, 5 + NUM_CLASSES])

    # Get predictions
    pred_xy = tf.sigmoid(pred[..., 0:2])  # 0-1 offset within cell
    pred_wh = pred[..., 2:4]               # log scale
    pred_obj = tf.sigmoid(pred[..., 4:5])
    pred_cls = pred[..., 5:]

    # Box loss (only for cells with objects)
    xy_loss = tf.reduce_sum(tf.square(pred_xy - target_xy) * obj_mask[..., tf.newaxis])
    wh_loss = tf.reduce_sum(tf.square(pred_wh - target_wh) * obj_mask[..., tf.newaxis])
    box_loss = (xy_loss + wh_loss) / (tf.reduce_sum(obj_mask) + 1e-7)

    # Objectness loss
    obj_loss = tf.keras.losses.binary_crossentropy(target_obj[..., tf.newaxis], pred_obj)
    obj_weight = obj_mask * 5.0 + (1.0 - obj_mask) * 0.5
    obj_loss = tf.reduce_mean(obj_loss * obj_weight)

    # Class loss
    target_onehot = tf.one_hot(tf.cast(target_classes, tf.int32), NUM_CLASSES)
    cls_loss = tf.keras.losses.categorical_crossentropy(target_onehot, pred_cls, from_logits=True)
    cls_loss = tf.reduce_sum(cls_loss * obj_mask) / (tf.reduce_sum(obj_mask) + 1e-7)

    return 5.0 * box_loss + obj_loss + cls_loss

print("Loss function defined")

Loss function defined


In [22]:
# Block 10: Training with Progress Bar (FRESH START - LOCAL STORAGE)

!pip install tqdm -q

from tqdm import tqdm
import time

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
train_losses = []
valid_losses = []
best_loss = float('inf')

@tf.function
def train_step(images, targets_list):
    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        loss = 0.0
        for i in range(3):
            loss += compute_loss(
                predictions[i],
                targets_list[i][0],
                targets_list[i][1],
                targets_list[i][2],
                targets_list[i][3],
                targets_list[i][4],
                i
            )
        loss = loss / 3.0

    grads = tape.gradient(loss, model.trainable_variables)
    grads = [tf.clip_by_norm(g, 10.0) if g is not None else g for g in grads]
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def prepare_targets_for_batch(targets):
    boxes = targets["boxes"].numpy()
    classes = targets["classes"].numpy()
    num_boxes = targets["num_boxes"].numpy()

    targets_list = []
    for scale_idx in range(3):
        t_obj, t_xy, t_wh, t_cls, mask = create_targets(boxes, classes, num_boxes, scale_idx)
        targets_list.append([
            tf.constant(t_obj, dtype=tf.float32),
            tf.constant(t_xy, dtype=tf.float32),
            tf.constant(t_wh, dtype=tf.float32),
            tf.constant(t_cls, dtype=tf.int32),
            tf.constant(mask, dtype=tf.float32)
        ])
    return targets_list

def validate():
    losses = []
    for i in range(len(valid_gen)):
        images, targets = valid_gen[i]
        images = tf.constant(images)
        targets = {k: tf.constant(v) for k, v in targets.items()}
        targets_list = prepare_targets_for_batch(targets)
        preds = model(images, training=False)
        loss = 0.0
        for j in range(3):
            loss += compute_loss(preds[j], targets_list[j][0], targets_list[j][1],
                                targets_list[j][2], targets_list[j][3], targets_list[j][4], j)
        losses.append(float(loss / 3.0))
    return np.mean(losses)

# Training with progress bar
print("=" * 60)
print("🚀 TRAINING STARTED (FRESH - WITH LOCAL STORAGE)")
print(f"Dataset: {len(train_data)} train, {len(valid_data)} valid")
print(f"Batch Size: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"📍 Data source: LOCAL STORAGE (FAST!)")
print("=" * 60)

total_batches = len(train_gen)
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    epoch_losses = []

    # Progress bar for batches
    pbar = tqdm(
        range(total_batches),
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        bar_format="{l_bar}{bar:30}{r_bar}",
        colour="green"
    )

    for batch_idx in pbar:
        images, targets = train_gen[batch_idx]
        images = tf.constant(images)
        targets = {k: tf.constant(v) for k, v in targets.items()}
        targets_list = prepare_targets_for_batch(targets)

        loss = train_step(images, targets_list)
        epoch_losses.append(float(loss))

        # Update progress bar
        avg_loss = np.mean(epoch_losses)
        pbar.set_postfix({
            "loss": f"{float(loss):.4f}",
            "avg": f"{avg_loss:.4f}"
        })

    train_gen.on_epoch_end()

    # Validation
    print("Validating...", end=" ")
    valid_loss = validate()

    train_loss = np.mean(epoch_losses)
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    epoch_time = time.time() - epoch_start
    total_time = time.time() - start_time

    # Print epoch summary
    print(f"Done!")
    print(f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f} | Time: {epoch_time:.1f}s")

    # Save best model
    if valid_loss < best_loss:
        best_loss = valid_loss
        model.save_weights(f'{CHECKPOINT_DIR}/best_weights.weights.h5')
        print(f"New best model saved! (Valid Loss: {valid_loss:.4f})")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.save_weights(f'{CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.weights.h5')
        print(f"Checkpoint saved: epoch_{epoch+1}")

    # ETA calculation
    avg_epoch_time = total_time / (epoch + 1)
    remaining_epochs = EPOCHS - (epoch + 1)
    eta = avg_epoch_time * remaining_epochs
    eta_min = int(eta // 60)
    eta_sec = int(eta % 60)
    print(f" ETA: {eta_min}m {eta_sec}s | Epoch speed: {epoch_time:.1f}s")
    print()

# Training complete summary
total_time = time.time() - start_time
hours = int(total_time // 3600)
mins = int((total_time % 3600) // 60)
secs = int(total_time % 60)
# print("=" * 60)
# print("🎉 TRAINING COMPLETE!")
print(f"Total Time: {hours}h {mins}m {secs}s")
print(f"Best Valid Loss: {best_loss:.4f}")
print(f"Best model saved to: {CHECKPOINT_DIR}/best_weights.weights.h5")
# print("=" * 60)

🚀 TRAINING STARTED (FRESH - WITH LOCAL STORAGE)
Dataset: 27841 train, 2614 valid
Batch Size: 16 | Epochs: 60 | LR: 0.001
Checkpoints: /content/drive/MyDrive/SmartVisionAI/checkpoints
📍 Data source: LOCAL STORAGE (FAST!)


Epoch 1/60: 100%|██████████████████████████████| 1740/1740 [08:56<00:00,  3.24it/s, loss=13.4795, avg=7.3347]


Validating... Done!
Train Loss: 7.3347 | Valid Loss: 7.0232 | Time: 586.4s
New best model saved! (Valid Loss: 7.0232)
 ETA: 576m 37s | Epoch speed: 586.4s



Epoch 2/60: 100%|██████████████████████████████| 1740/1740 [08:39<00:00,  3.35it/s, loss=4.2103, avg=5.2997]


Validating... Done!
Train Loss: 5.2997 | Valid Loss: 5.6409 | Time: 568.9s
New best model saved! (Valid Loss: 5.6409)
 ETA: 558m 33s | Epoch speed: 568.9s



...

Validating... Done!
Train Loss: 0.2075 | Valid Loss: 2.5245 | Time: 565.9s
 ETA: 18m 58s | Epoch speed: 565.9s



Epoch 59/60: 100%|██████████████████████████████| 1740/1740 [08:35<00:00,  3.37it/s, loss=0.2350, avg=0.2030]


Validating... Done!
Train Loss: 0.2030 | Valid Loss: 2.5147 | Time: 564.7s
 ETA: 9m 29s | Epoch speed: 564.7s



Epoch 60/60: 100%|██████████████████████████████| 1740/1740 [08:36<00:00,  3.37it/s, loss=0.1435, avg=0.1956]


Validating... Done!
Train Loss: 0.1956 | Valid Loss: 2.6097 | Time: 565.6s
Checkpoint saved: epoch_60
 ETA: 0m 0s | Epoch speed: 565.6s

Total Time: 9h 29m 11s
Best Valid Loss: 2.3453
Best model saved to: /content/drive/MyDrive/SmartVisionAI/checkpoints/best_weights.weights.h5


In [ ]:
# Block 11: Save Model (MISSING BEFORE - NOW ADDED)

print("Saving final model...")

# Load best weights
model.load_weights(f'{CHECKPOINT_DIR}/best_weights.weights.h5')

# Save complete model
model.save(f"{OUTPUT_DIR}/smart_vision_model.keras")

# Save class names
with open(f"{OUTPUT_DIR}/class_names.txt", 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')

# Save training history
with open(f"{OUTPUT_DIR}/history.pkl", 'wb') as f:
    pickle.dump({'train': train_losses, 'valid': valid_losses}, f)

# Save config
with open(f"{OUTPUT_DIR}/config.txt", 'w') as f:
    f.write(f"IMG_SIZE={IMG_SIZE}\n")
    f.write(f"NUM_CLASSES={NUM_CLASSES}\n")
    f.write(f"BATCH_SIZE={BATCH_SIZE}\n")
    f.write(f"EPOCHS={EPOCHS}\n")
    f.write(f"BEST_LOSS={best_loss}\n")

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train')
plt.plot(valid_losses, label='Valid')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)
plt.savefig(f"{OUTPUT_DIR}/training_history.png")
plt.show()

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("Files: smart_vision_model.keras, class_names.txt, history.pkl, config.txt, training_history.png")

In [ ]:
# Block 12: Test Inference

def detect(image_path, threshold=0.3):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = img.shape[:2]

    input_img = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    input_img = input_img.astype(np.float32) / 255.0
    input_img = np.expand_dims(input_img, 0)

    preds = model.predict(input_img, verbose=0)
    boxes, scores, classes = [], [], []

    for scale_idx, pred in enumerate(preds):
        grid = GRID_SIZES[scale_idx]
        pred = pred[0].reshape(grid, grid, 3, 5 + NUM_CLASSES)

        for r in range(grid):
            for c in range(grid):
                for a in range(3):
                    obj = 1 / (1 + np.exp(-pred[r, c, a, 4]))
                    if obj < threshold:
                        continue

                    tx, ty = pred[r, c, a, 0:2]
                    tw, th = pred[r, c, a, 2:4]

                    x = (c + 1/(1+np.exp(-tx))) / grid
                    y = (r + 1/(1+np.exp(-ty))) / grid
                    w = ANCHORS[scale_idx][a][0] * np.exp(min(tw, 5))
                    h = ANCHORS[scale_idx][a][1] * np.exp(min(th, 5))

                    cls_probs = 1 / (1 + np.exp(-pred[r, c, a, 5:]))
                    cls_id = np.argmax(cls_probs)
                    conf = obj * cls_probs[cls_id]

                    if conf >= threshold:
                        boxes.append([max(0,x-w/2), max(0,y-h/2), min(1,x+w/2), min(1,y+h/2)])
                        scores.append(conf)
                        classes.append(cls_id)

    colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), (255,0,255),
              (0,255,255), (128,0,0), (0,128,0), (255,128,0), (128,0,128)]

    for box, score, cls in zip(boxes, scores, classes):
        x1, y1 = int(box[0] * orig_w), int(box[1] * orig_h)
        x2, y2 = int(box[2] * orig_w), int(box[3] * orig_h)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), colors[cls], 2)
        cv2.putText(img_rgb, f"{CLASS_NAMES[cls]}: {score:.2f}", (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[cls], 2)

    return img_rgb, list(zip(classes, scores, boxes))

print("Testing model...")
test_dir = f"{MERGED_DIR}/test/images"
test_files = os.listdir(test_dir)[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, fname in enumerate(test_files):
    result, detections = detect(f"{test_dir}/{fname}")
    axes[idx // 3, idx % 3].imshow(result)
    axes[idx // 3, idx % 3].set_title(f"{len(detections)} detections")
    axes[idx // 3, idx % 3].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/test_results.png")
plt.show()

print(f"Test results saved to: {OUTPUT_DIR}/test_results.png")
print("Done!")
